> Onnx related functionality will be created here

In [ ]:
#| default_exp onnx_func

In [ ]:
#| export
import onnx
import numpy as np
from onnx import helper, numpy_helper
import onnxruntime as ort
import tf2onnx
import cv2
from typing import List, Tuple, Dict, Any
from tqdm.auto import tqdm
import shutil

In [ ]:
#| export
import tensorflow as tf
from tensorflow.keras.layers import Input, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Conv2DTranspose, concatenate, BatchNormalization, Activation, Concatenate

In [ ]:
#| export
from cv_tools.core import *
from pathlib import Path
import os
from typing import Tuple, List, Union

In [ ]:
import  os
#os.environ['HF_HOME'] = '/home/ai_warstein/data/huggingface'

In [ ]:
find_contours_binary

In [ ]:
def batch_img(x:np.ndarray): return np.expand_dims(x, axis=-1)[None, ...]

## Getting masks 

In [ ]:
HOME=os.getenv('HOME')
data_p = os.getenv('DATA_PATH')
data_p_easy = Path(data_p, 'easy_pin_detection')

#data_p=Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped')
msk_data_path = Path(rf'{data_p_easy}/addition_pin_sn_images_generated_masks')
im_data_path = Path(rf'{data_p_easy}/addition_pin_sn_images_generated_images')
#im_data_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/time_14_10_28_val_frGrnd0.9702_epoch_200.h5/failed/additional/addition_pin_sn_images_generated_images')
masks = msk_data_path.ls()
sm_msk = masks[0]
name_ = sm_msk.name

msk = read_img(sm_msk)
img = read_img(Path(im_data_path, name_))
show_(msk)

## Getting contours

In [ ]:
from transformers import SegGptImageProcessor, SegGptForImageSegmentation

checkpoint = "BAAI/seggpt-vit-large"
image_processor = SegGptImageProcessor.from_pretrained(checkpoint)
model = SegGptForImageSegmentation.from_pretrained(
    checkpoint)
    #force_download=True,
    #resume_download=False)

In [ ]:
! pip show labeling_test

In [ ]:
def frm_im_msk_to_sn_pin(
    img:OpenCvImage,
    msk:OpenCvImage,
    des_h:int,
    des_w:int,
    sn_pin_img_save_path:Path=None,
    sn_pin_msk_save_path:Path=None,
	name_:str=None,
	idx_:int=None
    ):


    cntrs = find_contours_binary(msk)
    back_offset = 40
    for idx, cntr in enumerate(cntrs):
        x,y,w,h = frm_cntr_to_bbox(cntr)
        offset_h = (des_h - h) - back_offset
        offset_w = (des_w - w)

        # indexing for height
        start_idx_h = y-back_offset
        end_idx_h = y + h + offset_h

        # indexing for width
        start_idx_w = x
        end_idx_w = x + w + offset_w

        cur_img =img[start_idx_h:end_idx_h, start_idx_w:end_idx_w] 
        n_h, n_w = cur_img.shape[:2]
    
        if n_h < 128:
            remain_h = 128 - n_h
            start_idx_h = start_idx_h - remain_h
        if n_w < 128:
            remain_w = 128 - n_w
            start_idx_w = start_idx_w - remain_w
        cur_img = img[start_idx_h:end_idx_h, start_idx_w:end_idx_w]
        cur_msk = msk[start_idx_h:end_idx_h, start_idx_w:end_idx_w]

        if idx_ is not None:
            name_ = f"{idx_}_{name_}"
        if sn_pin_img_save_path is not None:
            Path(sn_pin_img_save_path).mkdir(parents=True, exist_ok=True)
            cv2.imwrite(f"{str(sn_pin_img_save_path)}/cntr_{idx}_{name_}", cur_img)
        if sn_pin_msk_save_path is not None:
            Path(sn_pin_msk_save_path).mkdir(parents=True, exist_ok=True)
            cv2.imwrite(f"{str(sn_pin_msk_save_path)}/cntr_{idx}_{name_}", cur_msk)

# Paste image to another image


In [ ]:
'
cloned_img = cv2.seamlessClone(
        replace_part,
        full_img,
        mask,
        center,
        cv2.NORMAL_CLONE)

In [ ]:
def paste_image(
        im_src:OpenCvImage,# source image
		msk_src:OpenCvImage,# source mask to make sure I am not pasting another image to the background not foreground
		im_dst:OpenCvImage,# destination image, which will be pasted to src iamge
		im_save_path:Path=None,
		im_name:str=None,
		show:bool=False,
		clone_method:str='mixed'
        ):
	' Seamless clone the dst image to the src image'


	allowed_mask = cv2.bitwise_not(msk_src)
	show_(allowed_mask)

	cntrs = find_contours_binary(allowed_mask)
	
	if cntrs:
		largest_cntr = max(cntrs, key=cv2.contourArea)
		x, y, w, h = frm_cntr_to_bbox(largest_cntr)
		print(x, y, w, h)

		# check if the image is too small to be pasted

		if im_dst.shape[0] <= h or im_dst.shape[1] <= w:
			center = (x + w//2, y + h//2)
			print(center)
			res = seamless_clone(im_src, im_dst, center,
						clone_method=clone_method)
			if im_save_path is not None:
				Path(im_save_path).mkdir(parents=True, exist_ok=True)
				cv2.imwrite(f"{str(im_save_path)}/{im_name}", res)
			if show: show_(res)


In [ ]:
model_name = 'time_22_06_46_val_frGrnd0.9519_epoch_171.h5'
im_dst_p = Path(
    rf'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/{model_name}/failed/additional/addition_pin_sn_images')
im_names = im_dst_p.ls()

im_src = Path(r'/home/i4a_dev/data/projects/easy_pin_detection/trn_images_patch_256_min_px_350')
msk_src = Path(r'/home/i4a_dev/data/projects/easy_pin_detection/trn_masks_patch_256_min_px_350')

print(f"{len(im_names)}")
sm_im = im_names[0]
read_img(sm_im).shape
msk_src.ls()

In [ ]:
REF_IM_PATH="/home/i4a_dev/data/projects/easy_pin_detection/ref_images_patch_250"
REF_MSK_PATH="/home/i4a_dev/data/projects/easy_pin_detection/ref_masks_patch_250"
IM_DST="/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/time_22_06_46_val_frGrnd0.9519_epoch_171.h5/failed/additional/addition_pin_sn_images"

In [ ]:
ref_ims = Path(REF_IM_PATH).ls()
sm_im = ref_ims[0]
src_img = read_img(sm_im)
sm_dst_im = Path(IM_DST).ls()[0]
dst_img = read_img(sm_dst_im)
name = sm_im.name
sm_msk =  Path(rf'{REF_MSK_PATH}/{name}')
src_msk = read_img(sm_msk)
show_(sm_im)

In [ ]:
show_(dst_img)

In [ ]:
src_img_shape = src_img.shape
dst_img_shape = dst_img.shape

In [ ]:
mask_coords = np.column_stack(np.where(src_msk==0))
for y, x in mask_coords:
    # Check if the small image can fit at the current position
    if x + dst_img_shape[1] <= src_img_shape[1] and y + dst_img_shape[0] <= src_img_shape[0]:
        # Paste the small image onto the large image
        center_x = x + dst_img_shape[1] // 2
        center_y = y + dst_img_shape[0] // 2

        if src_msk[y:y+dst_img_shape[0], x:x+dst_img_shape[1]].sum() == 0:
            new_img = seamless_clone(
                src_img, 
                dst_img, (center_x, center_y),
                clone_method='normal')

            src_img[y:y+dst_img_shape[0], x:x+dst_img_shape[1]] = dst_img
            break



In [ ]:
show_(src_img)

In [ ]:
show_(new_img)

In [ ]:
src_img.shape

In [ ]:
src_img

In [ ]:
paste_image(
    im_src=read_img(sm_im),
    msk_src=read_img(sm_msk),
    im_dst=read_img(sm_dst_im),
    im_name=sm_im.name,
    show=True)

In [ ]:
paste_image(
    im_src=read_img(sm_im),
    msk_src=read_img(sm_msk),
    im_dst=read_img(sm_dst_im),
    im_name=sm_im.name,
    show=True,clone_method='normal')

In [ ]:
ref_im_dst = Path(im_src.parent,'ref_images_patch_250')
ref_msk_dst = Path(im_src.parent,'ref_masks_patch_250')
ref_im_dst.mkdir(exist_ok=True, parents=True), ref_msk_dst.mkdir(exist_ok=True, parents=True)

In [ ]:
#| export
def get_common_files(im_path, msk_path):
    im_files = get_name_(im_path.ls())
    print(f' Number of images: {len(im_files)}')
    msk_files = get_name_(msk_path.ls())
    print(f' Number of masks: {len(msk_files)}')
    common_files = set(im_files).intersection(set(msk_files))
    print(f' Number of common files: {len(common_files)}')
    return common_files

In [ ]:
trn_images = np.random.choice(im_src.ls(), 200)

In [ ]:

for i in tqdm(trn_images, total=len(trn_images)):
    name_ = i.name

    shutil.copyfile(i, f'{ref_im_dst}/{name_}')
    shutil.copyfile(f'{msk_src}/{name_}', f'{ref_msk_dst}/{name_}')

In [ ]:
frm_im_msk_to_sn_pin(img, msk, 128, 128, Path(data_p_easy, 'gen_image'), Path(data_p_easy, 'gen_mask'))

In [ ]:
#im_path = data_p.ls()
#s_im_path = im_path[0]
#img = read_img(s_im_path)
#img_n = (img / 255.0).astype(np.float32)
#b_img =batch_img(img_n)
#b_img.shape

In [ ]:
#| export
def split_image_to_patches(x, patch_size=128):
    # Calculate dynamic padding if needed
    _, height, width, _ = x.shape
    pad_height = (patch_size - height % patch_size) % patch_size
    pad_width = (patch_size - width % patch_size) % patch_size
    x = tf.pad(x, [[0, 0], [0, pad_height], [0, pad_width], [0, 0]], mode='CONSTANT')
    print(x.shape)
    # Reshape into patches
    resized_height, resized_width = x.shape[1], x.shape[2]
    x = tf.reshape(x, (-1, resized_height // patch_size, patch_size, resized_width // patch_size, patch_size, 1))
    x = tf.transpose(x, [0, 1, 3, 2, 4, 5])
    return tf.reshape(x, (-1, patch_size, patch_size, 1))

In [ ]:
#| export
def reconstruct_from_patches(
        patches, 
        original_shape, 
        patch_size=128):
    # Assuming patches are square and the batch dimension is flat
    num_patches_per_row = tf.math.ceil(tf.cast(original_shape[1], float) / patch_size)
    num_patches_per_col = tf.math.ceil(tf.cast(original_shape[2], float) / patch_size)


    # Reshape to grid
    patches = tf.reshape(patches, (-1, num_patches_per_row, num_patches_per_col, patch_size, patch_size, original_shape[-1]))
    patches = tf.transpose(patches, [0, 1, 3, 2, 4, 5])
    patches = tf.reshape(patches, (-1, num_patches_per_row * patch_size, num_patches_per_col * patch_size, original_shape[-1]))
    
    # Crop to original dimensions
    return patches[:, :original_shape[1], :original_shape[2], :]

In [ ]:
input_shape = (128, 128, 1)
input_shape = (1,) + input_shape
input_shape

In [ ]:
#| export
def create_onnx_model_patch(
    input_im_shape:Tuple[int, int, int],
    model:tf.keras.models.Model, # Normal model without pre and post processing
    onnx_m_name:str, # Name of the onnx model
    onnx_m_save_path:str, # where to save the  models
    patch_size:int=128

    ):
    'Create onnx model for for patching'
    inputs = Input(shape=input_im_shape)
    patches = Lambda(
        lambda x: split_image_to_patches(
                            x, patch_size=patch_size
                            ))(inputs)
    batch_output = model(patches)
    outputs = Lambda(
        lambda x: reconstruct_from_patches(
                            x, (1,) + input_im_shape, patch_size=patch_size,
                            ))(batch_output)
    new_model = Model(inputs=inputs, outputs=outputs)

    if onnx_m_name is None:
        onnx_m_name = f"converted_model.onnx"

    onnx_model, _ = tf2onnx.convert.from_keras(
        new_model, opset=13, output_path=f"{onnx_m_save_path}/{onnx_m_name}")
    return onnx_model

In [ ]:
m_path=Path(r'/home/ai_sintercra.work/data/projects/easy_pin_detection/models/patch_model_min_100_pixel/time_21_10_47_val_frGrnd0.9525_epoch_174.h5')
onnx_m_save_path='/home/ai_sintercra.work/data/projects/easy_pin_detection/models/patch_model_min_100_pixel'

model = tf.keras.models.load_model(f'{m_path}', compile=False)

In [ ]:
input_im_shape=(1152, 1632, 1)
patch_size=128
inputs = Input(shape=input_im_shape)
patches = Lambda(
        lambda x: split_image_to_patches(
                            x, patch_size=patch_size
                            ))(inputs)
        
batch_output = model(patches)
outputs = Lambda(
        lambda x: reconstruct_from_patches(
                            x, (1,)+ input_im_shape, patch_size=patch_size
                            ))(batch_output)
new_model = Model(inputs=inputs, outputs=outputs)
print(new_model.summary())

In [ ]:
onnx_m_name = Path(m_path).stem

In [ ]:
HEIGHT=1152
WIDTH=1632
CHANNELS=1
onnx_model = create_onnx_model_patch(
    input_im_shape=(HEIGHT, WIDTH, CHANNELS),
    model=model,
    onnx_m_name=f"{onnx_m_name}.onnx",
    onnx_m_save_path=Path(r'/home/ai_sintercra.work/data/projects/easy_pin_detection/models/patch_model_min_100_pixel'),
    patch_size=128
)

In [ ]:
onnx_m_name = Path(m_path).stem

In [ ]:
onnx_m_save_path

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
input_shape = (1, 1152, 1632, 1)
input_data = np.random.rand(*input_shape).astype(np.float32)
print(input_data.shape)
options = ort.SessionOptions()
options.intra_op_num_threads = 4
session = ort.InferenceSession(f"{onnx_m_save_path}/{onnx_m_name}.onnx", options)
output = session.run(None, {'input_8': input_data}) 

In [ ]:
output[0].shape

In [ ]:
#| export
def patching(
        img,
        patch_size:int=128):
        patch_size_ = [1, patch_size, patch_size, 1]
        strides = [1, patch_size, patch_size, 1]

        patches_e = tf.image.extract_patches(
                images=tf.expand_dims(img[None,...], axis=-1), rates=[1, 1, 1, 1], sizes=patch_size_, strides=strides, padding='VALID')
        
        patch_dim = patches_e.shape[1] * patches_e.shape[2]
        print(f'aft patching: {patches_e.shape}')
        print(patch_dim)
        patches_r = tf.reshape(patches_e, [patch_dim, patch_size, patch_size, 1])

        print(f'aft patching reshape: {patches_r.shape}')
        return patches_r


In [ ]:
#| export
def reconstruc_from_patches(
        pred_img, 
        img_s:Tuple[int, int, int],
        patch_size:int=128

    ):
    
    n_p_h = img_s[0] // patch_size
    n_p_w = img_s[1] // patch_size


    pred_img_r = tf.reshape(pred_img, [n_p_h, n_p_w, patch_size, patch_size])

    r_img = np.zeros(img_s, dtype=np.float32)

    n_patches_h = n_p_h
    n_patches_w = n_p_w

    for i in range(n_patches_h):
        for j in range(n_patches_w):
            r_img[i * patch_size: (i+1)*patch_size, j * patch_size: (j+1)*patch_size] = pred_img_r[i, j]
    return r_img

In [ ]:
#| export
def from_im_to_patch_pred(
        im_path:Union[str, Path], # image path 
        m_path:Union[str, Path], # model path with filename
        patch_size:int=128,
		threshold:float=0.9

        )->Tuple[np.ndarray, np.ndarray, np.ndarray]: # return image, normalized image, reconstructed mask
   ' Read, normalize image and patch it and batch it and create prediction and reconstruct it'
   img = read_img(im_path)
   # Normalize
   img_n = img / 255.0
   # patching
   p_img = patching(img_n, patch_size=patch_size )
   # loading model
   model = tf.keras.models.load_model(model_path, compile=False)

   # prediction
   pred_img = model.predict(p_img)
   pred_im_t = pred_img > threshold
   r_img = reconstruc_from_patches(pred_im_t, img.shape, patch_size=128)
   return img,img_n, r_img




In [ ]:
_, _, msk = from_im_to_patch_pred(s_im_path, m_path, patch_size=128, threshold=0.9)

In [ ]:
n_p = pred_img>  0.9

In [ ]:
import cv2

In [ ]:
n_p.shape
for i in range(len(n_p)):
    px = cv2.countNonZero(n_p[i].astype(np.uint8))
    if px > 10:
        show_(n_p[i])
    

In [ ]:
show_(pred_img[6])

In [ ]:
show_(img)

In [ ]:
show_(r_img)

In [ ]:
r_img.shape

In [ ]:
def conv_block(input_tensor, num_filters):
    """Function to add 2 convolutional layers with the parameters passed to it"""
    # first layer
    x = Conv2D(filters=num_filters, kernel_size=(3, 3), padding="same")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    
    # second layer
    x = Conv2D(filters=num_filters, kernel_size=(3, 3), padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    
    return x

In [ ]:
def encoder_block(input_tensor, num_filters):
    """Function to add 2 convolutional layers with the parameters passed to it and then perform max pooling"""
    x = conv_block(input_tensor, num_filters)
    p = MaxPooling2D((2, 2))(x)
    return x, p

def decoder_block(input_tensor, concat_tensor, num_filters):
    """Function to upsample, and add concatenate the upsampled tensor and a passed tensor"""
    x = Conv2DTranspose(num_filters, (2, 2), strides=(2, 2), padding="same")(input_tensor)
    x = concatenate([x, concat_tensor], axis=-1)
    x = conv_block(x, num_filters)
    return x

def build_unet(input_shape):
    inputs = Input(input_shape)
    
    # Encoder
    c1, p1 = encoder_block(inputs, 64)
    c2, p2 = encoder_block(p1, 128)
    c3, p3 = encoder_block(p2, 256)
    c4, p4 = encoder_block(p3, 512)
    
    # Bridge
    b1 = conv_block(p4, 1024)
    
    # Decoder
    d1 = decoder_block(b1, c4, 512)
    d2 = decoder_block(d1, c3, 256)
    d3 = decoder_block(d2, c2, 128)
    d4 = decoder_block(d3, c1, 64)
    
    # Output
    outputs = Conv2D(1, (1, 1), activation="sigmoid")(d4)
    
    model = Model(inputs=[inputs], outputs=[outputs])
    return model

In [ ]:
# Build U-Net model
input_shape = (576, 816, 3)
unet_model = build_unet(input_shape)
unet_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
unet_model.summary()

In [ ]:
height, width, channels = 1152, 1632, 1
image = np.random.rand(1, height, width, channels).astype(np.float32)

In [ ]:
height, width, channels = 1152, 1632, 1
image = tf.cast(tf.random.uniform(shape=[height, width, channels], minval=0, maxval=256, dtype=tf.int32), tf.float32)
image.shape

In [ ]:
x = image
height, width, channels = tf.shape(x)

# Pad the images to have dimensions divisible by 128
padded_height = tf.math.ceil(height / 128) * 128
padded_width = tf.math.ceil(width / 128) * 128
#x_padded = tf.pad(x, [[0, 0], [0, padded_height - height], [0, padded_width - width], [0, 0]])

In [ ]:
#| export
def split_image_to_patches(x, patch_size=128):
    # Calculate dynamic padding if needed
    _, height, width, _ = x.shape
    pad_height = (patch_size - height % patch_size) % patch_size
    pad_width = (patch_size - width % patch_size) % patch_size
    x = tf.pad(x, [[0, 0], [0, pad_height], [0, pad_width], [0, 0]], mode='CONSTANT')
    print(x.shape)
    # Reshape into patches
    resized_height, resized_width = x.shape[1], x.shape[2]
    x = tf.reshape(x, (-1, resized_height // patch_size, patch_size, resized_width // patch_size, patch_size, 1))
    x = tf.transpose(x, [0, 1, 3, 2, 4, 5])
    return tf.reshape(x, (-1, patch_size, patch_size, 1))

In [ ]:
#| export
def reconstruct_from_patches(patches, original_shape, patch_size=128.0):
    # Assuming patches are square and the batch dimension is flat
    num_patches_per_row = tf.math.ceil(tf.cast(original_shape[1], float) / patch_size)
    num_patches_per_col = tf.math.ceil(tf.cast(original_shape[2], float) / patch_size)


    # Reshape to grid
    patches = tf.reshape(patches, (-1, num_patches_per_row, num_patches_per_col, patch_size, patch_size, original_shape[-1]))
    patches = tf.transpose(patches, [0, 1, 3, 2, 4, 5])
    patches = tf.reshape(patches, (-1, num_patches_per_row * patch_size, num_patches_per_col * patch_size, original_shape[-1]))
    
    # Crop to original dimensions
    return patches[:, :original_shape[1], :original_shape[2], :]

In [ ]:
def create_onnx_model_patch(
    input_im_shape:Tuple[int, int, int],
    model:tf.keras.models.Model,
    m_path:Path,
    onnx_m_name:str,
    patch_size:int=128

    ):
    'Create onnx model for for patching'
    inputs = Input(shape=input_im_shape)
    patches = Lambda(
        lambda x: split_image_to_patches(
                            x, patch_size=patch_size
                            ))(inputs)
    batch_output = model(patches)
    outputs = Lambda(
        lambda x: reconstruct_from_patches(
                            x, input_im_shape, patch_size=128
                            ))(batch_output)
    model = Model(inputs=inputs, outputs=outputs)

    if onnx_m_name is None:
        onnx_m_name = f"converted_model.onnx"

    onnx_model, _ = tf2onnx.convert.from_keras(
        model, opset=13, output_path=f"{m_path}/{onnx_m_name}")
    return model

In [ ]:

    inputs = Input(shape=(1152, 1632, 1))  #
    patches = Lambda(lambda x: split_image_to_patches(x, patch_size=128))(inputs)
    batch_output = model(patches)
    outputs = Lambda(lambda x: reconstruct_from_patches(x, inputs.shape, patch_size=128))(batch_output)
    model = Model(inputs=inputs, outputs=outputs)

In [ ]:
im_path 
img = read_img(s_im_path)
img_n = img / 255.0
img_n = np.expand_dims(img_n, axis=-1)[None,...]

In [ ]:
m_path = Path(data_p, 'easy_pin_detection/models/patch_125_models')
m_f_name = m_path.ls()[1]

In [ ]:

model = tf.keras.models.load_model(f'{m_f_name}', compile=False, safe_mode=False)

In [ ]:
patches = split_image_to_patches(img_n)
pred_patches = model.predict(patches)
rec_msk = reconstruct_from_patches(pred_patches, img_n.shape, patch_size=128)
rec_msk.shape

In [ ]:
inputs = Input(shape=(1152, 1632, 1))  #
patches = Lambda(lambda x: split_image_to_patches(x, patch_size=128))(inputs)
batch_output = model(patches)
outputs = Lambda(lambda x: reconstruct_from_patches(x, inputs.shape, patch_size=128))(batch_output)
model = Model(inputs=inputs, outputs=outputs)

In [ ]:
m_base_path = m_path.parent

In [ ]:
tf.keras.models.save_model(model, f'{m_path}/unet_model_patch.keras')

In [ ]:
load_model_ = tf.keras.models.load_model(f'{m_path}/unet_model_patch.keras', safe_mode=False)

In [ ]:
#Convert TensorFlow model to ONNX
onnx_model, _ = tf2onnx.convert.from_keras(load_model_, opset=13, output_path=f"{m_path}/converted_model.onnx")

In [ ]:
img_n_ = img_n.astype(np.float32)

In [ ]:
import onnxruntime as ort
import numpy as np

# Load the ONNX model
session = ort.InferenceSession(f"{m_path}/converted_model.onnx")
img = img_n_
input_name = session.get_inputs()[0].name
outputs = session.run(None, {input_name: img}) 

In [ ]:
outputs[0].shape

In [ ]:
import time

start_time = time.time()
num_runs = 100
for _ in range(num_runs):
    session.run(None, {input_name: img})
elapsed_time = time.time() - start_time

print(f"Average inference time per run: {elapsed_time / num_runs} seconds")

In [ ]:
show_(outputs[0][0])

In [ ]:
pred_ = model.predict(img_n)
show_(pred_[0])

In [ ]:
new_img_n = new_img / 255.0

In [ ]:
model.predict(new_img_n[0])

In [ ]:
new_img[0].shape

In [ ]:
r_img_ = reconstruct_from_patches(new_p, new_img.shape, patch_size=128)

In [ ]:
img = read_img(s_im_path)
img = np.expand_dims(img, -1)
new_img =img[None]

In [ ]:
new_img.shape

In [ ]:
new_p = split_image_to_patches(new_img, patch_size=128)

In [ ]:
model.save('split_model.h5')

In [ ]:
import tf2onnx

# Convert
spec = (
    tf.TensorSpec((None, None, None, 3), 
                  tf.float32, 
                  name="input"),)
output_path = "image_splitter.onnx"
model_proto, external_tensor_storage = tf2onnx.convert.from_keras(
    model, 
    input_signature=spec, 
    opset=13, 
    output_path=output_path)

In [ ]:
session = ort.InferenceSession('image_splitter.onnx')

In [ ]:
# Get input name for the model
input_name = session.get_inputs()[0].name

# Run the model
output = session.run(None, {input_name: image})

In [ ]:
len(output)

In [ ]:
output[0].shape

In [ ]:
show_(output[0])